# Actividad Didáctica 2 - Módulo 2: Simulación de Procesos Estocásticos
## Modelo Oculto de Markov (HMM) para la Predicción del Tráfico Vehicular

**Integrantes del Equipo:**
- Camilo Palacios y compañeros de equipo

**Objetivo:**
Aplicar el conocimiento de las distribuciones de probabilidad y cadenas de Markov mediante la implementación de un **Modelo Oculto de Markov (HMM)** que permita simular y predecir la dinámica estocástica del tráfico vehicular en un corredor urbano arterial.

### 1. Importación de Librerías y Carga del Módulo HMM

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from traffic_hmm.model import TrafficHMM
from traffic_hmm.simulation import TrafficSimulation
from traffic_hmm.visualizations import TrafficVisualizer

# Instanciar el modelo
hmm = TrafficHMM()
print('Estados Ocultos:', hmm.states)
print('Observaciones Visibles:', hmm.observations)
print('Vector Inicial pi:', hmm.pi)

### 2. Definición y Visualización de las Matrices del HMM
- **Matriz de Transición de Estados ($A$):** Modela la inercia del tráfico y probabilidad de cambiar de estado.
- **Matriz de Emisión de Sensores ($B$):** Modela la probabilidad de que un radar mida cierta velocidad dado el estado real.

In [ ]:
viz = TrafficVisualizer(output_dir='results')
viz.plot_matrices(hmm)

# Mostrar la matriz A con pandas
df_A = pd.DataFrame(hmm.A, index=hmm.states, columns=hmm.states)
print('--- Matriz de Transición A ---')
display(df_A)

# Mostrar la matriz B con pandas
df_B = pd.DataFrame(hmm.B, index=hmm.states, columns=hmm.observations)
print('--- Matriz de Emisión B ---')
display(df_B)

### 3. Propiedades Analíticas Estacionarias y Tiempos de Permanencia
Resolvemos el sistema lineal $\pi^* A = \pi^*$ con $\sum \pi^*_i = 1$ para hallar el régimen de largo plazo.

In [ ]:
pi_star = hmm.compute_stationary_distribution()
holding_times = hmm.expected_holding_times()

print('Distribución Estacionaria Teórica:')
for s, p in zip(hmm.states, pi_star):
    print(f'  • {s:14s}: {p*100:6.2f}% ({p:.4f})')

print('\nTiempo Promedio de Permanencia Continua (Holding Time):')
for s, h in holding_times.items():
    print(f'  • {s:14s}: {h:.2f} pasos ({h*15:.1f} minutos)')

# Graficar convergencia desde condiciones iniciales extremas
viz.plot_stationary_convergence(hmm, steps=15)
plt.show()

### 4. Simulación de una Jornada de 24 Horas y Decodificación de Viterbi
Se simulan 96 intervalos de 15 minutos. El algoritmo de Viterbi reconstruye el estado oculto basándose únicamente en las lecturas ruidosas de los radares de velocidad.

In [ ]:
sim = TrafficSimulation(hmm=hmm, step_duration_min=15)
res_24h = sim.run_single_day_simulation(num_steps=96, seed=42, start_hour=6)

df_24h = res_24h['dataframe']
print(f'Exactitud de Viterbi: {res_24h["accuracy"]*100:.2f}%')
display(df_24h.head(10))

# Visualizar trayectoria temporal
viz.plot_single_day_timeline(res_24h)
viz.plot_confusion_matrix(res_24h['confusion_matrix'], hmm.states)
plt.show()

### 5. Validación Monte Carlo (1,000 Réplicas = 96,000 Intervalos)
Demostración de la Ley de los Grandes Números: convergencia de las frecuencias simuladas hacia la distribución estacionaria teórica.

In [ ]:
mc_res = sim.run_monte_carlo_validation(num_replicas=1000, steps_per_replica=96, seed=100)

print(f'MAE Global (Teórico vs Empírico): {mc_res["mae_stationary"]:.6f}')
print(f'Exactitud Promedio Viterbi: {mc_res["mean_accuracy"]*100:.2f}% (±{mc_res["std_accuracy"]*100:.2f}%)')

viz.plot_monte_carlo_comparison(mc_res, hmm.states)
plt.show()

### 6. Escenario de Incidente y Resiliencia del Sistema
Simulación de un accidente o lluvia intensa en la ventana horaria entre 14:00 y 17:00.

In [ ]:
incident_res = sim.run_perturbation_scenario(num_steps=96, incident_start=32, incident_end=44, seed=77)
viz.plot_incident_scenario(incident_res)
plt.show()

### 7. Conclusiones Principales
1. **Modelado Estocástico Efectivo:** El HMM captura con fidelidad la inercia del tráfico urbano, donde las transiciones no son arbitrarias sino gobernadas por probabilidades condicionales realistas.
2. **Potencia de la Inferencia Oculta:** A pesar de que los sensores presentan un margen de error del 20-35%, el algoritmo de Viterbi logra una exactitud de reconstrucción superior al 73%, demostrando la ventaja de incorporar la memoria secuencial de la matriz de transición.
3. **Convergencia Asintótica:** Las 1,000 réplicas Monte Carlo confirmaron la convergencia hacia la distribución estacionaria teórica con un MAE inferior a 0.008, validando la estabilidad ergódica del proceso.